# Imbalance, the newsvendor and quantile decisions

A retailer buys a volume `q` for each hour in the day-ahead market. The actual load `L` turns
out different. Being short means buying the shortfall at an expensive imbalance price; being
long means selling the surplus at a cheap one. How much should you buy?

Everything is shown first on numbers you can check by hand, then on the hourly data.

**What's in here**
- the cost of one decision, step by step
- the newsvendor rule: buy a quantile, not the mean
- five scenarios, five candidate volumes: the full cost table
- what a quantile forecast is, on 8 points
- pinball loss by hand
- honest features: what is known at 12:00 the day before
- quantile regression, calibration and pinball loss on real data
- policies compared in money: point forecast, safety margin, quantile, perfect foresight
- the fractile moves with the market
- value of information: what a weather forecast is worth in euros

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, QuantileRegressor

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
np.set_printoptions(precision=4, suppress=True)

## 1. The cost of one decision

One hour. Day-ahead price 100, system-buy price 130 (what you pay if short), system-sell
price 90 (what you receive if long). Actual load turns out to be 10 MWh.

First: you bought `q = 8`, so you are 2 MWh short.

In [2]:
p_da, p_buy, p_sell = 100.0, 130.0, 90.0
L = 10.0

q = 8.0
short = max(L - q, 0)
surplus = max(q - L, 0)
cost = q * p_da + short * p_buy - surplus * p_sell
print("short  :", short, "MWh")
print("surplus:", surplus, "MWh")
print("cost   :", q, "*", p_da, "+", short, "*", p_buy, "-", surplus, "*", p_sell, "=", cost)

short  : 2.0 MWh
surplus: 0 MWh
cost   : 8.0 * 100.0 + 2.0 * 130.0 - 0 * 90.0 = 1060.0


If you had known `L` exactly you would have bought 10 at 100 → 1000. The extra 60 is the
**imbalance cost**: 2 MWh short × (130 − 100) = 2 × 30.

Now the other side: you bought `q = 12`, so you are 2 MWh long.

In [3]:
q = 12.0
short = max(L - q, 0)
surplus = max(q - L, 0)
cost = q * p_da + short * p_buy - surplus * p_sell
print("short  :", short, " surplus:", surplus)
print("cost   :", cost, " -> excess over perfect foresight:", cost - L * p_da)

short  : 0  surplus: 2.0
cost   : 1020.0  -> excess over perfect foresight: 20.0


2 MWh long × (100 − 90) = 20. So per MWh:

- `c_u` = cost of being **under** (short) = `p_buy − p_da` = 30
- `c_o` = cost of being **over** (long) = `p_da − p_sell` = 10

Being short is three times as expensive as being long. That asymmetry is the whole story.

In [4]:
c_u = p_buy - p_da
c_o = p_da - p_sell
print("c_u:", c_u, " c_o:", c_o)

c_u: 30.0  c_o: 10.0


## 2. The newsvendor rule on five scenarios

Suppose the load will be one of `[8, 9, 10, 11, 12]`, each equally likely, and `c_u = 3`,
`c_o = 1`. For every candidate volume `q`, the excess cost in each scenario is

`c_u · max(L − q, 0) + c_o · max(q − L, 0)`

In [5]:
scenarios = np.array([8, 9, 10, 11, 12])
c_u, c_o = 3.0, 1.0
candidates = np.array([8, 9, 10, 11, 12])

table = pd.DataFrame(index=pd.Index(scenarios, name="scenario L"), columns=pd.Index(candidates, name="q"))
for q in candidates:
    for L in scenarios:
        table.loc[L, q] = c_u * max(L - q, 0) + c_o * max(q - L, 0)
table = table.astype(float)
table

q,8,9,10,11,12
scenario L,,,,,
8,0.0,1.0,2.0,3.0,4.0
9,3.0,0.0,1.0,2.0,3.0
10,6.0,3.0,0.0,1.0,2.0
11,9.0,6.0,3.0,0.0,1.0
12,12.0,9.0,6.0,3.0,0.0


Read one cell: `q = 9`, `L = 12` → 3 MWh short × 3 = 9. Or `q = 12`, `L = 8` → 4 long × 1 = 4.

Each scenario is equally likely, so the expected cost of a `q` is the column mean.

In [6]:
expected = table.mean(axis=0)
print(expected)
print()
print("best q:", expected.idxmin(), " expected cost:", expected.min())

q
8     6.0
9     3.8
10    2.4
11    1.8
12    2.0
dtype: float64

best q: 11  expected cost: 1.8


The best volume is 11, **above** the mean load of 10. The rule that predicts this:

`τ* = c_u / (c_u + c_o)` and `q* = the τ*-quantile of L`.

In [7]:
tau_star = c_u / (c_u + c_o)
print("tau*:", tau_star)
print("quantile(scenarios, tau*):", np.quantile(scenarios, tau_star))
print("mean(scenarios)          :", scenarios.mean(), " -> expected cost at the mean:", expected[10])

tau*: 0.75
quantile(scenarios, tau*): 11.0
mean(scenarios)          : 10.0  -> expected cost at the mean: 2.4


0.75 → the 75th percentile of {8,9,10,11,12} is 11. Buying the mean (10) costs 2.0 per hour
instead of 1.6.

**Pitfall:** swap `c_u` and `c_o` and τ* becomes 0.25 → q* = 9: you would systematically
under-buy and pay the expensive side. Write down which cost is which before coding.

In [8]:
tau_wrong = c_o / (c_u + c_o)
print("tau if swapped:", tau_wrong, " -> q =", np.quantile(scenarios, tau_wrong), " expected cost:", expected[9])

tau if swapped: 0.25  -> q = 9.0  expected cost: 3.8


## 3. What a quantile forecast is, on 8 points

The mean forecast (OLS) draws a line through the middle. A **0.9-quantile forecast** draws a
line with 90 % of the points below it. `QuantileRegressor(quantile=τ, alpha=0)` fits it by
minimising the pinball loss (`alpha=0`: no penalty; the default `alpha=1` shrinks hard).

In [9]:
x8 = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=float)
y8 = np.array([1, 3, 2, 5, 4, 8, 5, 9], dtype=float)
X8 = x8.reshape(-1, 1)          # sklearn wants a 2-D feature matrix

fits = pd.DataFrame({"x": x8, "y": y8})
fits["ols"] = LinearRegression().fit(X8, y8).predict(X8)
for tau in [0.1, 0.5, 0.9]:
    qr = QuantileRegressor(quantile=tau, alpha=0, solver="highs").fit(X8, y8)
    fits[f"q{int(tau*100)}"] = qr.predict(X8)
fits.round(2)

,x,y,ols,q10,q50,q90
0,1.0,1.0,1.17,0.50,1.00,1.75
1,2.0,3.0,2.15,1.25,2.14,3.00
2,3.0,2.0,3.14,2.00,3.29,4.25
3,4.0,5.0,4.13,2.75,4.43,5.50
4,5.0,4.0,5.12,3.50,5.57,6.75
5,6.0,8.0,6.11,4.25,6.71,8.00
6,7.0,5.0,7.10,5.00,7.86,9.25
7,8.0,9.0,8.08,5.75,9.00,10.50


In [10]:
print("share of y below q10:", (y8 <= fits["q10"]).mean())
print("share of y below q50:", (y8 <= fits["q50"]).mean())
print("share of y below q90:", (y8 <= fits["q90"]).mean())

share of y below q10: 0.25
share of y below q50: 0.625
share of y below q90: 1.0


That share is the **coverage**: a τ-quantile forecast should have about τ of the outcomes at
or below it. With only 8 points the lines pass through data points and the shares are rough
(the 0.9 line ends up above everything). On 4,000 rows the coverage should land near τ, and
checking that is the first thing to do with a quantile forecast.

### Pinball loss by hand

For a τ-quantile forecast, an error `e = actual − forecast` costs `τ · e` if positive and
`(τ − 1) · e` if negative. Four numbers, τ = 0.9:

In [11]:
tau = 0.9
actual = np.array([10.0, 10.0, 10.0, 10.0])
forecast = np.array([8.0, 9.0, 11.0, 12.0])
e = actual - forecast
loss = np.maximum(tau * e, (tau - 1) * e)
pd.DataFrame({"actual": actual, "forecast": forecast, "e": e, "tau*e": tau * e, "(tau-1)*e": (tau - 1) * e, "pinball": loss})

,actual,forecast,e,tau*e,(tau-1)*e,pinball
0,10.0,8.0,2.0,1.8,-0.2,1.8
1,10.0,9.0,1.0,0.9,-0.1,0.9
2,10.0,11.0,-1.0,-0.9,0.1,0.1
3,10.0,12.0,-2.0,-1.8,0.2,0.2


Under-forecasting by 2 costs 1.8; over-forecasting by 2 costs only 0.2. A forecast that
minimises this loss sits high, on purpose. That is why RMSE is the wrong score for a
0.9-quantile: it would call the deliberate upward bias an error.

## 4. Real data: imbalance prices

The file has only the day-ahead price. Build a buy price `DA + premium` and a sell price
`DA − premium`, premiums log-normal so they are positive and sometimes large, with the buy
premium larger on average.

In [12]:
rng = np.random.default_rng(42)
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
SHARE = 0.05                                   # a retailer with 5 % of national load
df["load_mwh"] = df["consumption_mwh"] * SHARE
df["p_da"] = df["price_eur_mwh"]
df["p_buy"] = df["p_da"] + rng.lognormal(np.log(25), 0.6, len(df))
df["p_sell"] = df["p_da"] - rng.lognormal(np.log(12), 0.6, len(df))
df[["time", "load_mwh", "p_da", "p_buy", "p_sell"]].head()

,time,load_mwh,p_da,p_buy,p_sell
0,2022-01-01 00:00:00+00:00,1342.920,81.83,111.845265,64.018003
1,2022-01-01 01:00:00+00:00,1308.890,88.21,101.605052,76.949888
2,2022-01-01 02:00:00+00:00,1311.470,84.71,123.928420,69.422731
3,2022-01-01 03:00:00+00:00,1269.065,70.92,114.877122,54.648571
4,2022-01-01 04:00:00+00:00,1261.150,60.78,68.534356,45.997903


In [13]:
print("mean c_u (p_buy - p_da):", round((df["p_buy"] - df["p_da"]).mean(), 1))
print("mean c_o (p_da - p_sell):", round((df["p_da"] - df["p_sell"]).mean(), 1))

mean c_u (p_buy - p_da): 29.9
mean c_o (p_da - p_sell): 14.5


## 5. Honest features: what is known at 12:00 the day before

The volume for every hour of day D is decided at 12:00 on D−1 (the day-ahead auction).
For a target hour, the decision time is:

In [14]:
target = pd.Timestamp("2023-01-10 16:00", tz="UTC")
decision = target.floor("D") - pd.Timedelta(hours=12)
print("target  :", target)
print("decision:", decision)

target  : 2023-01-10 16:00:00+00:00
decision: 2023-01-09 12:00:00+00:00


Which lags are known at that moment? "Same hour yesterday" (lag 24) for a 16:00 target is
16:00 on D−1, which is *after* the 12:00 decision. Not known. Lag 48 and lag 168 are.

In [15]:
for lag in [24, 48, 168]:
    lag_time = target - pd.Timedelta(hours=lag)
    print(f"lag {lag:3d}: {lag_time}  known at decision time? {lag_time <= decision}")

lag  24: 2023-01-09 16:00:00+00:00  known at decision time? False
lag  48: 2023-01-08 16:00:00+00:00  known at decision time? True
lag 168: 2023-01-03 16:00:00+00:00  known at decision time? True


Weather: use the **forecast** available at the decision time, not the temperature that
actually happened. Three forecasts exist for our target hour; pick the latest one issued at or
before 12:00 on D−1.

In [16]:
fc = pd.read_csv("../data/weather_forecasts.csv", parse_dates=["origin_datetime", "forecast_datetime"])
three = fc[fc["forecast_datetime"] == target].sort_values("origin_datetime")
three

,origin_datetime,forecast_datetime,horizon_h,temp_forecast_c
35847,2023-01-09 00:00:00+00:00,2023-01-10 16:00:00+00:00,40,1.52
35883,2023-01-09 12:00:00+00:00,2023-01-10 16:00:00+00:00,28,-5.38
35919,2023-01-10 00:00:00+00:00,2023-01-10 16:00:00+00:00,16,1.10
35955,2023-01-10 12:00:00+00:00,2023-01-10 16:00:00+00:00,4,-0.06


In [17]:
allowed = three[three["origin_datetime"] <= decision]
print(allowed[["origin_datetime", "horizon_h", "temp_forecast_c"]])
print()
print("use the latest allowed origin:", allowed["origin_datetime"].max(), "-> horizon", allowed["horizon_h"].min(), "h")

                origin_datetime  horizon_h  temp_forecast_c
35847 2023-01-09 00:00:00+00:00         40             1.52
35883 2023-01-09 12:00:00+00:00         28            -5.38

use the latest allowed origin: 2023-01-09 12:00:00+00:00 -> horizon 28 h


`merge_asof` does exactly this for every row: match each decision time to the latest origin at
or before it, within the same target hour (`by="time"`).

In [18]:
df["decision_time"] = df["time"].dt.floor("D") - pd.Timedelta(hours=12)
fc_ren = fc.rename(columns={"forecast_datetime": "time"}).sort_values("origin_datetime")
m = pd.merge_asof(df.sort_values("decision_time"), fc_ren,
                  left_on="decision_time", right_on="origin_datetime", by="time", direction="backward")
m = m.sort_values("time").reset_index(drop=True)
m[["time", "decision_time", "origin_datetime", "horizon_h", "temp_c", "temp_forecast_c"]].iloc[[30, 36, 47]]

,time,decision_time,origin_datetime,horizon_h,temp_c,temp_forecast_c
30,2022-01-02 06:00:00+00:00,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,18.0,0.73,1.85
36,2022-01-02 12:00:00+00:00,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,24.0,6.04,5.24
47,2022-01-02 23:00:00+00:00,2022-01-01 12:00:00+00:00,2022-01-01 12:00:00+00:00,35.0,2.57,1.48


In [19]:
ok = m.dropna(subset=["origin_datetime"])
print("all origins <= decision time:", bool((ok["origin_datetime"] <= ok["decision_time"]).all()))
print("horizons used:", int(ok["horizon_h"].min()), "to", int(ok["horizon_h"].max()), "hours")

all origins <= decision time: True
horizons used: 12 to 35 hours


Now the feature table: legal lags, calendar, forecast heating/cooling degrees. `hdd_act`
uses the actual temperature and is kept only for the "cheating" comparison at the end.

In [20]:
m["hour"] = m["time"].dt.hour
m["is_weekend"] = (m["time"].dt.dayofweek >= 5).astype(int)
m["lag48"] = m["load_mwh"].shift(48)
m["lag168"] = m["load_mwh"].shift(168)
m["hdd_fc"] = np.maximum(15 - m["temp_forecast_c"], 0)
m["cdd_fc"] = np.maximum(m["temp_forecast_c"] - 22, 0)
m["hdd_act"] = np.maximum(15 - m["temp_c"], 0)
m["cdd_act"] = np.maximum(m["temp_c"] - 22, 0)
hour_dummies = pd.get_dummies(m["hour"], prefix="h").astype(float)
m = pd.concat([m, hour_dummies], axis=1)
HCOLS = list(hour_dummies.columns)
FEATS = ["lag48", "lag168", "is_weekend", "hdd_fc", "cdd_fc"] + HCOLS

data = m.dropna(subset=FEATS + ["load_mwh"]).reset_index(drop=True)
train = data[data["time"] < "2023-07-01"]
test = data[data["time"] >= "2023-07-01"]
print("train rows:", len(train), " test rows:", len(test))
train[["time", "load_mwh", "lag48", "lag168", "hdd_fc", "cdd_fc"]].head(3)

train rows: 12936  test rows: 4416


,time,load_mwh,lag48,lag168,hdd_fc,cdd_fc
0,2022-01-08 00:00:00+00:00,1410.525,1540.400,1342.92,16.00,0.0
1,2022-01-08 01:00:00+00:00,1310.265,1502.990,1308.89,17.75,0.0
2,2022-01-08 02:00:00+00:00,1306.350,1464.105,1311.47,17.33,0.0


## 6. Quantile regression on the real data

Fit τ = 0.1, 0.5, 0.9 and an OLS point model on the train period; predict the test period.

In [21]:
y = "load_mwh"
qpred = {}
for tau in [0.1, 0.5, 0.9]:
    qr = QuantileRegressor(quantile=tau, alpha=0, solver="highs").fit(train[FEATS], train[y])
    qpred[tau] = qr.predict(test[FEATS])
ols = LinearRegression().fit(train[FEATS], train[y])
pt_test = ols.predict(test[FEATS])
pt_train = ols.predict(train[FEATS])

pd.DataFrame({"actual": test[y].values, "q10": qpred[0.1], "q50": qpred[0.5], "q90": qpred[0.9], "ols": pt_test},
             index=test["time"]).head(6).round(0)

,actual,q10,q50,q90,ols
time,,,,,
2023-07-01 00:00:00+00:00,1042.0,1010.0,1065.0,1108.0,1062.0
2023-07-01 01:00:00+00:00,956.0,950.0,1006.0,1053.0,1003.0
2023-07-01 02:00:00+00:00,926.0,923.0,976.0,1028.0,976.0
2023-07-01 03:00:00+00:00,917.0,922.0,970.0,1021.0,971.0
2023-07-01 04:00:00+00:00,964.0,930.0,981.0,1036.0,981.0
2023-07-01 05:00:00+00:00,995.0,988.0,1040.0,1092.0,1041.0


Coverage on the test period, overall and by hour of day (a quantile can be right on average
and wrong at 18:00).

In [22]:
for tau in [0.1, 0.5, 0.9]:
    print(f"coverage of q{int(tau*100)}: {(test[y].values <= qpred[tau]).mean():.3f}")

coverage of q10: 0.111
coverage of q50: 0.537
coverage of q90: 0.922


In [23]:
below = pd.DataFrame({"q10": test[y].values <= qpred[0.1],
                      "q50": test[y].values <= qpred[0.5],
                      "q90": test[y].values <= qpred[0.9]}, index=test["hour"].values)
below.groupby(level=0).mean().round(2).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
q10,0.14,0.08,0.08,0.09,0.11,0.09,0.10,0.12,0.11,0.11,0.12,0.16,0.13,0.09,0.08,0.10,0.10,0.10,0.12,0.11,0.14,0.14,0.10,0.14
q50,0.60,0.57,0.60,0.57,0.58,0.53,0.51,0.51,0.57,0.51,0.55,0.58,0.51,0.49,0.52,0.53,0.55,0.49,0.52,0.46,0.57,0.51,0.55,0.52
q90,0.90,0.92,0.95,0.92,0.93,0.90,0.93,0.91,0.92,0.93,0.92,0.90,0.95,0.91,0.89,0.92,0.92,0.91,0.92,0.93,0.93,0.93,0.92,0.95


Pinball loss and RMSE for each model. Each quantile model should win its own column.

In [24]:
def pinball(y_true, y_pred, tau):
    e = y_true - y_pred
    return np.mean(np.maximum(tau * e, (tau - 1) * e))

rows = []
for name, pred in [("q10", qpred[0.1]), ("q50", qpred[0.5]), ("q90", qpred[0.9]), ("ols", pt_test)]:
    rmse = np.sqrt(np.mean((test[y].values - pred) ** 2))
    rows.append([name, rmse, pinball(test[y].values, pred, 0.1), pinball(test[y].values, pred, 0.5), pinball(test[y].values, pred, 0.9)])
pd.DataFrame(rows, columns=["model", "rmse", "pinball_0.1", "pinball_0.5", "pinball_0.9"]).round(1)

,model,rmse,pinball_0.1,pinball_0.5,pinball_0.9
0,q10,69.0,7.7,28.9,50.1
1,q50,44.5,19.2,17.5,15.9
2,q90,77.1,58.0,32.9,7.9
3,ols,44.5,19.3,17.5,15.8


q90 has the worst RMSE and the best pinball_0.9; q10 likewise wins its own column. Judge a
forecast by the loss the decision actually has.

**Pitfall:** quantiles fitted separately can cross (q90 below q50 on some rows). Check.

In [25]:
crossing = (qpred[0.9] < qpred[0.5]) | (qpred[0.5] < qpred[0.1])
print("rows with crossing quantiles:", crossing.sum(), "of", len(crossing))

rows with crossing quantiles: 0 of 4416


## 7. Policies compared in money

Four purchasing policies on the test period, each producing a volume `q` per hour:

1. **point**: buy the OLS forecast
2. **point + margin**: add a fixed margin, chosen on the *train* period
3. **quantile at τ\***: the quantile forecast at the critical fractile, τ\* from *train* premiums
4. **perfect foresight**: buy exactly `L` (the unreachable floor)

Cost per hour as in section 1, summed over the test period.

In [26]:
def total_cost(q, L, p_da, p_buy, p_sell):
    short = np.maximum(L - q, 0)
    surplus = np.maximum(q - L, 0)
    return (q * p_da + short * p_buy - surplus * p_sell).sum()

L_te = test[y].values
c_u_tr = (train["p_buy"] - train["p_da"]).mean()
c_o_tr = (train["p_da"] - train["p_sell"]).mean()
tau_star = c_u_tr / (c_u_tr + c_o_tr)
print("train c_u:", round(c_u_tr, 1), " c_o:", round(c_o_tr, 1), " -> tau*:", round(tau_star, 3))

train c_u: 29.8  c_o: 14.6  -> tau*: 0.671


In [27]:
# margin tuned on TRAIN: try a few and keep the cheapest
L_tr = train[y].values
rows = []
for margin in [-50, -25, 0, 25, 50, 75, 100]:
    cost_tr = total_cost(pt_train + margin, L_tr, train["p_da"].values, train["p_buy"].values, train["p_sell"].values)
    rows.append([margin, cost_tr / 1e6])
margins = pd.DataFrame(rows, columns=["margin MWh", "train cost M€"])
best_margin = margins.loc[margins["train cost M€"].idxmin(), "margin MWh"]
print(margins.round(3))
print("best margin on train:", best_margin)

   margin MWh  train cost M€
0         -50       2082.001
1         -25       2075.327
2           0       2071.378
3          25       2070.508
4          50       2072.228
5          75       2075.639
6         100       2079.902
best margin on train: 25


In [28]:
q_tau = QuantileRegressor(quantile=tau_star, alpha=0, solver="highs").fit(train[FEATS], train[y]).predict(test[FEATS])
perfect = total_cost(L_te, L_te, test["p_da"].values, test["p_buy"].values, test["p_sell"].values)

rows = []
for name, q in [("point (OLS)", pt_test), ("point + margin", pt_test + best_margin),
                (f"quantile @ tau*={tau_star:.2f}", q_tau), ("perfect foresight", L_te)]:
    tot = total_cost(q, L_te, test["p_da"].values, test["p_buy"].values, test["p_sell"].values)
    rows.append([name, tot / 1e6, (tot - perfect) / 1e6, np.mean(q > L_te)])
pd.DataFrame(rows, columns=["policy", "total cost M€", "excess over perfect M€", "share of hours long"]).round(3)

,policy,total cost M€,excess over perfect M€,share of hours long
0,point (OLS),524.110,3.343,0.540
1,point + margin,523.968,3.200,0.751
2,quantile @ tau*=0.67,523.912,3.144,0.708
3,perfect foresight,520.767,0.000,0.000


The margin and the quantile policy both beat the raw point forecast and land close together:
with a roughly constant error distribution a fixed margin *is* a crude quantile. The quantile
model earns its keep when uncertainty varies with the features.

**Interview check:** "You tuned the margin on the test set?" No, on train. Tuning on the period
you report is the same leak as tuning a hyperparameter on the test set.

## 8. The fractile moves with the market

Make the sell side the expensive one and τ\* drops below 0.5: buy *less* than the median.
Same forecast models, different premiums (deterministic here, to isolate the effect).

In [29]:
rows = []
for cu_, co_ in [(25, 12), (18, 18), (12, 25), (40, 8)]:
    tau = cu_ / (cu_ + co_)
    q = QuantileRegressor(quantile=tau, alpha=0, solver="highs").fit(train[FEATS], train[y]).predict(test[FEATS])
    pb_ = test["p_da"].values + cu_
    ps_ = test["p_da"].values - co_
    perf_ = total_cost(L_te, L_te, test["p_da"].values, pb_, ps_)
    ex_q = total_cost(q, L_te, test["p_da"].values, pb_, ps_) - perf_
    ex_pt = total_cost(pt_test, L_te, test["p_da"].values, pb_, ps_) - perf_
    rows.append([cu_, co_, round(tau, 2), ex_pt / 1e6, ex_q / 1e6, 1 - ex_q / ex_pt])
pd.DataFrame(rows, columns=["c_u", "c_o", "tau*", "excess: point M€", "excess: quantile M€", "saving"]).round(3)

,c_u,c_o,tau*,excess: point M€,excess: quantile M€,saving
0,25,12,0.68,2.738,2.597,0.052
1,18,18,0.50,2.785,2.789,-0.001
2,12,25,0.32,2.988,2.583,0.136
3,40,8,0.83,3.406,2.359,0.307


Symmetric costs: the quantile is the median and saves almost nothing. The more asymmetric the
market, the more a quantile decision is worth.

## 9. Value of information

Same decision rule, three feature sets: actual temperature at the target hour (impossible, a
ceiling), the honest forecast, no temperature at all. The cost differences are in euros.

In [30]:
variants = {
    "actual temperature (cheating ceiling)": ["lag48", "lag168", "is_weekend", "hdd_act", "cdd_act"] + HCOLS,
    "forecast temperature (honest)":         FEATS,
    "no temperature":                        ["lag48", "lag168", "is_weekend"] + HCOLS,
}
rows = []
for name, feats in variants.items():
    q = QuantileRegressor(quantile=tau_star, alpha=0, solver="highs").fit(train[feats], train[y]).predict(test[feats])
    ex = total_cost(q, L_te, test["p_da"].values, test["p_buy"].values, test["p_sell"].values) - perfect
    rows.append([name, ex / 1e6])
voi = pd.DataFrame(rows, columns=["features", "excess cost M€ (Jul-Dec 2023)"]).round(2)
voi

,features,excess cost M€ (Jul-Dec 2023)
0,actual temperature (cheating ceiling),2.70
1,forecast temperature (honest),3.14
2,no temperature,4.26


In [31]:
ex_act, ex_fc, ex_no = voi.iloc[0, 1], voi.iloc[1, 1], voi.iloc[2, 1]
print("value of the weather forecast           :", round(ex_no - ex_fc, 2), "M€ per half-year")
print("what a perfect temperature could still add:", round(ex_fc - ex_act, 2), "M€ (upper bound on any better forecast)")

value of the weather forecast           : 1.12 M€ per half-year
what a perfect temperature could still add: 0.44 M€ (upper bound on any better forecast)


**Interview check:** "Your RMSE improved by 3 %. So what?" Translate it: what did the decision
cost before and after, on the same test window, with the same prices?

## Quick reference

| Question | Answer |
|---|---|
| Volume under asymmetric imbalance costs | the `τ*`-quantile of load, `τ* = c_u / (c_u + c_o)`; `c_u` = cost of being short, `c_o` = cost of being long |
| Fit a conditional quantile | `QuantileRegressor(quantile=τ, alpha=0, solver="highs")` |
| Is the quantile any good | coverage on test ≈ τ, overall and by hour; pinball loss, not RMSE |
| Legal lags for a 12:00 D−1 decision | 48 h and 168 h, not 24 h; forecasts with origin ≤ decision time via `merge_asof` |
| Safety margin vs quantile | a margin tuned on train is a crude quantile; the model wins when uncertainty varies |
| Compare models | realised cost in € on the same window and prices; perfect foresight as the floor |
| Value of a data source | cost without it − cost with it; the cheating version bounds what is left |